In [2]:
import heapq
import random


SIZE = 70

grid = [[0 for _ in range(SIZE)] for _ in range(SIZE)]



density = input("Enter initial obstacle density (low/medium/high): ").lower()

if density == "low":
    obstacle_probability = 0.10

elif density == "medium":
    obstacle_probability = 0.20

elif density == "high":
    obstacle_probability = 0.30

else:
    print("Invalid density")
    exit()


for i in range(SIZE):
    for j in range(SIZE):

        if random.random() < obstacle_probability:
            grid[i][j] = 1



start_row = int(input("Enter start row (0-69): "))
start_col = int(input("Enter start column (0-69): "))

goal_row = int(input("Enter goal row (0-69): "))
goal_col = int(input("Enter goal column (0-69): "))

start = (start_row, start_col)
goal = (goal_row, goal_col)

# Start and goal must be free
grid[start_row][start_col] = 0
grid[goal_row][goal_col] = 0



def heuristic(node, goal):

    return abs(node[0] - goal[0]) + abs(node[1] - goal[1])




def a_star(grid, start, goal):

    queue = []

    # f(n) = g(n) + h(n)
    heapq.heappush(queue, (0, start))

    cost = {start: 0}

    parent = {start: None}

    nodes_explored = 0

    while queue:

        f, current = heapq.heappop(queue)

        nodes_explored += 1

        # Goal reached
        if current == goal:
            break

        row, col = current

        # Four possible movements
        directions = [
            (-1, 0),   # Up
            (1, 0),    # Down
            (0, -1),   # Left
            (0, 1)     # Right
        ]

        for dr, dc in directions:

            new_row = row + dr
            new_col = col + dc

            # Check boundaries
            if new_row < 0 or new_row >= SIZE:
                continue

            if new_col < 0 or new_col >= SIZE:
                continue

            # Check obstacle
            if grid[new_row][new_col] == 1:
                continue

            neighbour = (new_row, new_col)

            # Movement cost
            new_cost = cost[current] + 1

            # Better path found
            if neighbour not in cost or new_cost < cost[neighbour]:

                cost[neighbour] = new_cost

                h = heuristic(neighbour, goal)

                f = new_cost + h

                heapq.heappush(queue, (f, neighbour))

                parent[neighbour] = current



    if goal not in parent:

        return None, nodes_explored


    path = []

    current = goal

    while current is not None:

        path.append(current)

        current = parent[current]

    path.reverse()

    return path, nodes_explored


current = start

final_path = []

total_nodes_explored = 0

replanning_count = 0


while current != goal:

    # Find path from current position to goal
    path, nodes_explored = a_star(
        grid,
        current,
        goal
    )

    total_nodes_explored += nodes_explored

    # No path available
    if path is None:

        print("\nNo path exists!")

        print("UGV could not reach the goal.")

        break


    # Follow the path
    path_blocked = False

    for i in range(1, len(path)):

        next_position = path[i]



        if random.random() < 0.05:

            new_row = random.randint(0, SIZE - 1)

            new_col = random.randint(0, SIZE - 1)

            new_obstacle = (new_row, new_col)

            # Do not block current position or goal
            if new_obstacle != current and new_obstacle != goal:

                grid[new_row][new_col] = 1

                print(
                    "New obstacle detected at:",
                    new_obstacle)


        if grid[next_position[0]][next_position[1]] == 1:

            print(
                "Path blocked at:",
                next_position
            )

            print("Re-planning...\n")

            path_blocked = True

            replanning_count += 1

            break


        # Move UGV
        current = next_position

        final_path.append(current)


    # If path is blocked, calculate a new path
    if path_blocked:

        continue



if current == goal:

    print("\n===================================")
    print("UGV SUCCESSFULLY REACHED GOAL")
    print("===================================")

    print("Start:", start)

    print("Goal:", goal)

    print(
        "Total path length:",
        len(final_path),
        "units"
    )

    print(
        "Number of replanning operations:",
        replanning_count
    )

    print(
        "Total nodes explored:",
        total_nodes_explored
    )



final_path_set = set(final_path)

print("\nFinal UGV Path:\n")

for i in range(SIZE):

    row_output = ""

    for j in range(SIZE):

        position = (i, j)

        if position == start:

            row_output += "S "

        elif position == goal:

            row_output += "G "

        elif position in final_path_set:

            row_output += "* "

        elif grid[i][j] == 1:

            row_output += "# "

        else:

            row_output += ". "

    print(row_output)

Enter initial obstacle density (low/medium/high): low
Enter start row (0-69): 0
Enter start column (0-69): 0
Enter goal row (0-69): 20
Enter goal column (0-69): 20
New obstacle detected at: (7, 16)
New obstacle detected at: (37, 10)

UGV SUCCESSFULLY REACHED GOAL
Start: (0, 0)
Goal: (20, 20)
Total path length: 40 units
Number of replanning operations: 0
Total nodes explored: 348

Final UGV Path:

S * # . . . . . . # . . # . . . . . . . . . . . . . . . . . . . . . . # . . . # # . . . . # . . . . # . . . . . . . # . # . . . . . . . . . 
. * * * * * # . # . . . . . . . . . . . . . . . . . . . . # . . . . . . . . . . . . . # . . . . . . # . . # . . . . . . . # . . . # . . . . 
. . . . # * * * # . . . . . . . . . . . . # . . . . . . . . . . . . . . . . # . . . . . . . . . . . . . . . . . . . # . . . . # . . . . . . 
. . . # . . . * # . . . . # . . . . . # . . . . . . . # # . . . . . . . # . . . . . . . . . . . . . # . . # # . . # . . . . . . . . . . . . 
. . . . . . . * * * * * * * * * * * 